In [1]:

import re
import subprocess
from importlib import import_module
from pathlib import Path

import tomlkit

try:
    save_umap_json = import_module("hyrax.3d_viz.save_umap_to_json").save_umap_json
except Exception:
    save_umap_json = None


DEFAULT_BASE_DIRECTORY = Path("/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs")
DEFAULT_DATA_DIR = "/Users/diegomiura/research/Hyrax-Research/test_dir_100images/split_images"
DEFAULT_RESULTS_DIR = "/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/results/"
DEFAULT_FILTERS = ["g", "r", "i", "z", "y"]


def _default_output_dir(run_number, base_directory=DEFAULT_BASE_DIRECTORY):
    return Path(base_directory) / f"run{run_number}"


def _load_toml(path):
    with open(path, "r", encoding="utf-8") as handle:
        return tomlkit.load(handle)


def _write_toml(path, document):
    with open(path, "w", encoding="utf-8") as handle:
        handle.write(tomlkit.dumps(document))


def _create_local_job_content(log_filename, commands):
    command_block = "\n".join(commands)
    return f"""#!/usr/bin/env bash
set -euo pipefail

cd "$(dirname "$0")"
export PYTHONUNBUFFERED=1
JOB_NAME=$(basename "$0")

exec > "{log_filename}" 2>&1

echo "Starting ${{JOB_NAME}} at $(date)"
{command_block}
echo "Finished ${{JOB_NAME}} at $(date)"
"""


def _write_job_file(path, job_content):
    path.write_text(job_content, encoding="utf-8")
    path.chmod(0o755)


def _run_local_job_file(run_dir, job_file):
    run_dir = Path(run_dir)
    job_path = run_dir / job_file
    if not job_path.exists():
        raise FileNotFoundError(f"Job file not found: {job_path}")

    print(f"Running locally: bash {job_file}")
    subprocess.run(["bash", job_file], cwd=run_dir, check=True)


def _iter_dataset_wrappers(data_request_section):
    if not isinstance(data_request_section, dict):
        return
    for dataset_wrapper in data_request_section.values():
        if isinstance(dataset_wrapper, dict) and "data" in dataset_wrapper:
            yield dataset_wrapper


def _find_results_dir_by_run_name(results_root, run_name, expected_suffix=None):
    results_root = Path(results_root)
    if not results_root.exists():
        return None

    for candidate in sorted(results_root.iterdir(), reverse=True):
        if not candidate.is_dir():
            continue
        if expected_suffix and expected_suffix not in candidate.name:
            continue
        runtime_config = candidate / "runtime_config.toml"
        if not runtime_config.exists():
            continue
        try:
            config = _load_toml(runtime_config)
        except Exception:
            continue
        if config.get("train", {}).get("run_name") == run_name:
            return candidate

    return None


In [2]:

def create_training_files_v2(
    run_number,
    filter_catalog,
    crop_to=None,
    output_dir=None,
    data_dir=DEFAULT_DATA_DIR,
    results_dir=DEFAULT_RESULTS_DIR,
    model_name="HyraxAutoencoderV2",
    final_layer="tanh",
    latent_dim=64,
    base_channel_size=32,
    batch_size=256,
    epochs=20,
    dataset_class="FitsImageDataSet",
    object_id_column_name="object_id",
    filters=None,
):
    """Create training TOMLs and local shell job files for a run."""
    output_dir = Path(output_dir) if output_dir else _default_output_dir(run_number)
    crop_to = list(crop_to) if crop_to is not None else [150, 150]
    filters = list(filters) if filters is not None else list(DEFAULT_FILTERS)
    output_dir.mkdir(parents=True, exist_ok=True)

    optimizer_grid = [
        {"optimizer": "torch.optim.SGD", "lr": 0.01, "momentum": True},
        {"optimizer": "torch.optim.SGD", "lr": 0.1, "momentum": True},
        {"optimizer": "torch.optim.SGD", "lr": 0.001, "momentum": True},
        {"optimizer": "torch.optim.SGD", "lr": 0.0001, "momentum": True},
        {"optimizer": "torch.optim.Adam", "lr": 0.01, "momentum": False},
        {"optimizer": "torch.optim.Adam", "lr": 0.1, "momentum": False},
        {"optimizer": "torch.optim.Adam", "lr": 0.001, "momentum": False},
        {"optimizer": "torch.optim.Adam", "lr": 0.0001, "momentum": False},
    ]

    for file_number, optimizer_config in enumerate(optimizer_grid, start=1):
        create_toml_file_v2(
            run_number=run_number,
            file_number=file_number,
            optimizer_config=optimizer_config,
            filter_catalog=filter_catalog,
            crop_to=crop_to,
            output_dir=output_dir,
            data_dir=data_dir,
            results_dir=results_dir,
            model_name=model_name,
            final_layer=final_layer,
            latent_dim=latent_dim,
            base_channel_size=base_channel_size,
            batch_size=batch_size,
            epochs=epochs,
            dataset_class=dataset_class,
            object_id_column_name=object_id_column_name,
            filters=filters,
        )
        create_job_file_v2(run_number, file_number, output_dir)


def create_toml_file_v2(
    run_number,
    file_number,
    optimizer_config,
    filter_catalog,
    crop_to,
    output_dir,
    data_dir,
    results_dir,
    model_name,
    final_layer,
    latent_dim,
    base_channel_size,
    batch_size,
    epochs,
    dataset_class,
    object_id_column_name,
    filters,
):
    """Create a Hyrax runtime config that matches the current codebase."""
    optimizer_name = optimizer_config["optimizer"]
    run_name = f"run{run_number}_{file_number}"

    document = tomlkit.document()
    document["general"] = {
        "dev_mode": False,
        "log_level": "debug",
        "data_dir": str(data_dir),
        "results_dir": str(results_dir),
    }

    document["model"] = {
        "name": model_name,
        "base_channel_size": base_channel_size,
        "latent_dim": latent_dim,
        "final_layer": final_layer,
    }

    document["criterion"] = {
        "name": "torch.nn.MSELoss",
        "band_loss_reduction": "mean",
    }

    document["optimizer"] = {"name": optimizer_name}
    document[optimizer_name] = {"lr": optimizer_config["lr"]}
    if optimizer_config["momentum"]:
        document[optimizer_name]["momentum"] = 0.9

    document["train"] = {
        "weights_filename": "example_model.pth",
        "epochs": epochs,
        "resume": False,
        "split": "train",
        "experiment_name": f"run{run_number}",
        "run_name": run_name,
    }

    document["data_request"] = {
        "train": {
            "data": {
                "dataset_class": dataset_class,
                "data_location": str(data_dir),
                "fields": ["image"],
                "primary_id_field": "object_id",
            }
        },
        "infer": {
            "data": {
                "dataset_class": dataset_class,
                "data_location": str(data_dir),
                "fields": ["image"],
                "primary_id_field": "object_id",
            }
        },
    }

    document["data_set"] = {
        "name": dataset_class,
        "object_id_column_name": object_id_column_name,
        "use_cache": True,
        "preload_cache": True,
        "seed": 1,
        "train_size": 0.8,
        "validate_size": 0.1,
        "test_size": 0.1,
        "filter_catalog": str(filter_catalog),
        "filters": list(filters),
        "transform": "tanh",
        "crop_to": list(crop_to),
    }

    document["data_loader"] = {"batch_size": batch_size}

    output_path = Path(output_dir) / f"train{run_number}_{file_number}.toml"
    _write_toml(output_path, document)
    print(f"Created {output_path}")


def create_job_file_v2(run_number, file_number, output_dir):
    """Create a local shell job file for training."""
    output_dir = Path(output_dir)
    toml_path = output_dir / f"train{run_number}_{file_number}.toml"
    job_content = _create_local_job_content(
        log_filename=f"train{run_number}_{file_number}.txt",
        commands=[f"hyrax train --runtime-config={toml_path}"],
    )
    output_path = output_dir / f"train{run_number}_{file_number}.sh"
    _write_job_file(output_path, job_content)
    print(f"Created {output_path}")


def submit_training_jobs_v2(run_number, start_num, end_num, base_directory=DEFAULT_BASE_DIRECTORY):
    """Run a contiguous range of training jobs locally."""
    run_dir = Path(base_directory) / f"run{run_number}"
    for i in range(start_num, end_num + 1):
        job_file = f"train{run_number}_{i}.sh"
        _run_local_job_file(run_dir, job_file)


def copy_training_configs_v2(source_dir, target_dir, old_run_num, new_run_num, new_data_dir=None, new_filter_catalog_root=None):
    """Copy generated training configs/jobs and update run identifiers and key paths."""
    source_path = Path(source_dir)
    target_path = Path(target_dir)
    target_path.mkdir(parents=True, exist_ok=True)

    copied = 0
    for toml_file in sorted(source_path.glob(f"train{old_run_num}_*.toml")):
        suffix = toml_file.stem.split("_")[-1]
        new_toml_name = f"train{new_run_num}_{suffix}.toml"
        new_job_name = f"train{new_run_num}_{suffix}.sh"

        config = _load_toml(toml_file)
        config.setdefault("train", {})
        config["train"]["experiment_name"] = f"run{new_run_num}"
        config["train"]["run_name"] = f"run{new_run_num}_{suffix}"
        config["train"]["weights_filename"] = config["train"].get("weights_filename", "example_model.pth")
        config["general"]["log_level"] = "debug"

        if new_data_dir is not None:
            config["general"]["data_dir"] = str(new_data_dir)
            for split_name in ("train", "infer", "validate"):
                split_config = config.get("data_request", {}).get(split_name)
                for dataset_wrapper in _iter_dataset_wrappers(split_config):
                    dataset_wrapper["data"]["data_location"] = str(new_data_dir)

        if new_filter_catalog_root is not None and config.get("data_set", {}).get("filter_catalog"):
            current_name = Path(str(config["data_set"]["filter_catalog"])).name
            config["data_set"]["filter_catalog"] = str(Path(new_filter_catalog_root) / current_name)

        _write_toml(target_path / new_toml_name, config)

        job_file = source_path / f"train{old_run_num}_{suffix}.sh"
        if job_file.exists():
            job_content = job_file.read_text(encoding="utf-8")
            job_content = job_content.replace(f"train{old_run_num}_{suffix}.txt", f"train{new_run_num}_{suffix}.txt")
            job_content = job_content.replace(f"run{old_run_num}/train{old_run_num}_{suffix}.toml", f"run{new_run_num}/train{new_run_num}_{suffix}.toml")
            _write_job_file(target_path / new_job_name, job_content)

        copied += 1

    print(f"Copied {copied} training configurations from run{old_run_num} to run{new_run_num}")


def extract_model_directory_v2(train_output_file):
    """Return the trained weights path, using log parsing first and run-name matching as fallback."""
    train_output_file = Path(train_output_file)
    if not train_output_file.exists():
        raise FileNotFoundError(f"Training output file not found: {train_output_file}")

    content = train_output_file.read_text(encoding="utf-8", errors="replace")
    patterns = [
        r"Latest checkpoint saved as: (.+)/checkpoint_epoch_\d+\.pt",
        r"Best metric checkpoint saved as: (.+)/checkpoint_[^/\s]+\.pt",
        r"Exported model to ONNX format: (.+)/[^/\s]+\.onnx",
    ]
    for pattern in patterns:
        match = re.search(pattern, content)
        if match:
            return str(Path(match.group(1)) / "example_model.pth")

    sibling_toml = train_output_file.with_suffix(".toml")
    if sibling_toml.exists():
        config = _load_toml(sibling_toml)
        results_root = config.get("general", {}).get("results_dir")
        run_name = config.get("train", {}).get("run_name")
        weights_filename = config.get("train", {}).get("weights_filename", "example_model.pth")
        if results_root and run_name:
            results_dir = _find_results_dir_by_run_name(results_root, run_name, expected_suffix="-train-")
            if results_dir is not None:
                candidate = results_dir / weights_filename
                if candidate.exists():
                    return str(candidate)

    raise ValueError(f"Could not determine model weights path from {train_output_file}")


def create_infer_scripts_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY):
    """Generate infer TOMLs/jobs from sibling train files."""
    run_dir = Path(base_directory) / f"run{x}"
    results = []

    for y in y_values:
        train_name = f"train{x}_{y}"
        infer_name = f"infer{x}_{y}"
        train_toml = run_dir / f"{train_name}.toml"
        train_job = run_dir / f"{train_name}.sh"
        train_output = run_dir / f"{train_name}.txt"

        if not train_toml.exists():
            print(f"Warning: {train_toml} not found, skipping...")
            continue

        try:
            model_weights_path = extract_model_directory_v2(train_output)
        except (FileNotFoundError, ValueError) as exc:
            print(f"Error processing {train_name}: {exc}")
            continue

        infer_toml = run_dir / f"{infer_name}.toml"
        config = _load_toml(train_toml)
        config["infer"] = {"model_weights_file": str(model_weights_path), "split": False}
        _write_toml(infer_toml, config)

        if train_job.exists():
            infer_job = run_dir / f"{infer_name}.sh"
            job_content = train_job.read_text(encoding="utf-8")
            job_content = job_content.replace(f"train{x}_{y}.txt", f"infer{x}_{y}.txt")
            job_content = job_content.replace(f"train{x}_{y}.toml", f"infer{x}_{y}.toml")
            job_content = job_content.replace("hyrax train", "hyrax infer")
            _write_job_file(infer_job, job_content)
            results.append((infer_toml, infer_job))
            print(f"Created infer scripts for {train_name}")
        else:
            print(f"Warning: {train_job} not found for {train_name}")

    return results


def submit_infer_jobs_v2(run_number, job_numbers, base_directory=DEFAULT_BASE_DIRECTORY):
    """Run selected infer jobs locally."""
    run_dir = Path(base_directory) / f"run{run_number}"
    for i in job_numbers:
        job_file = f"infer{run_number}_{i}.sh"
        _run_local_job_file(run_dir, job_file)


def extract_inference_directory_v2(infer_output_file):
    """Extract the inference results directory from infer output logs."""
    infer_output_file = Path(infer_output_file)
    if not infer_output_file.exists():
        raise FileNotFoundError(f"Inference output file not found: {infer_output_file}")

    content = infer_output_file.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"Saving inference results at: (.+)", content)
    if match:
        return match.group(1).strip().rstrip("/") + "/"

    raise ValueError(f"Could not find inference directory in {infer_output_file}")


In [3]:

def create_udb_scripts_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY):
    """Generate UDB TOMLs/jobs from infer artifacts."""
    run_dir = Path(base_directory) / f"run{x}"
    results = []

    for y in y_values:
        infer_name = f"infer{x}_{y}"
        udb_name = f"udb{x}_{y}"
        infer_toml = run_dir / f"{infer_name}.toml"
        infer_job = run_dir / f"{infer_name}.sh"
        infer_output = run_dir / f"{infer_name}.txt"

        if not infer_toml.exists():
            print(f"Warning: {infer_toml} not found, skipping...")
            continue

        try:
            inference_dir = extract_inference_directory_v2(infer_output)
        except (FileNotFoundError, ValueError) as exc:
            print(f"Error processing {infer_name}: {exc}")
            continue

        udb_toml = run_dir / f"{udb_name}.toml"
        config = _load_toml(infer_toml)
        config["results"] = {"inference_dir": inference_dir}
        config["vector_db"] = {"name": "chromadb", "infer_results_dir": inference_dir}
        config["umap"] = {
            "fit_sample_size": 5000,
            "save_fit_umap": False,
            "parallel": True,
            "name": "umap.UMAP",
            "UMAP": {"n_components": 2, "n_neighbors": 15},
        }
        _write_toml(udb_toml, config)

        if infer_job.exists():
            udb_job = run_dir / f"{udb_name}.sh"
            job_content = infer_job.read_text(encoding="utf-8")
            job_content = job_content.replace(f"infer{x}_{y}.txt", f"udb{x}_{y}.txt")
            job_content = job_content.replace(f"infer{x}_{y}.toml", f"udb{x}_{y}.toml")
            input_dir = inference_dir.rstrip("/")
            replacement = (
                f"hyrax umap --runtime-config="
                f"{{runtime}} --input-dir={input_dir}/\n"
                f"hyrax save_to_database --runtime-config={{runtime}} --input-dir={input_dir}/"
            )
            match = re.search(r"hyrax infer --runtime-config=([^\s]+)", job_content)
            if match:
                job_content = job_content.replace(
                    f"hyrax infer --runtime-config={match.group(1)}",
                    replacement.format(runtime=match.group(1)),
                )
            _write_job_file(udb_job, job_content)
            results.append((udb_toml, udb_job))
            print(f"Created UDB scripts for {infer_name}")
        else:
            print(f"Warning: {infer_job} not found for {infer_name}")

    return results


def submit_udb_jobs_v2(run_number, job_numbers, base_directory=DEFAULT_BASE_DIRECTORY):
    """Run selected UDB jobs locally."""
    run_dir = Path(base_directory) / f"run{run_number}"
    for i in job_numbers:
        job_file = f"udb{run_number}_{i}.sh"
        _run_local_job_file(run_dir, job_file)


def create_3dumap_scripts_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY):
    """Generate 3D UMAP TOMLs/jobs from UDB artifacts."""
    run_dir = Path(base_directory) / f"run{x}"
    results = []

    for y in y_values:
        udb_name = f"udb{x}_{y}"
        dumap_name = f"3dumap{x}_{y}"
        udb_toml = run_dir / f"{udb_name}.toml"
        udb_job = run_dir / f"{udb_name}.sh"

        if not udb_toml.exists():
            print(f"Warning: {udb_toml} not found, skipping...")
            continue

        dumap_toml = run_dir / f"{dumap_name}.toml"
        config = _load_toml(udb_toml)
        config.setdefault("umap", {})
        config["umap"].setdefault("UMAP", {})
        config["umap"]["UMAP"]["n_components"] = 3
        _write_toml(dumap_toml, config)

        if udb_job.exists():
            dumap_job = run_dir / f"{dumap_name}.sh"
            lines = udb_job.read_text(encoding="utf-8").splitlines()
            filtered_lines = [line for line in lines if "hyrax save_to_database" not in line]
            job_content = "\n".join(filtered_lines) + "\n"
            job_content = job_content.replace(f"udb{x}_{y}.txt", f"3dumap{x}_{y}.txt")
            job_content = job_content.replace(f"udb{x}_{y}.toml", f"3dumap{x}_{y}.toml")
            _write_job_file(dumap_job, job_content)
            results.append((dumap_toml, dumap_job))
            print(f"Created 3D UMAP scripts for {udb_name}")
        else:
            print(f"Warning: {udb_job} not found for {udb_name}")

    return results


def submit_3dumap_jobs_v2(run_number, job_numbers, base_directory=DEFAULT_BASE_DIRECTORY):
    """Run selected 3D UMAP jobs locally."""
    run_dir = Path(base_directory) / f"run{run_number}"
    for i in job_numbers:
        job_file = f"3dumap{run_number}_{i}.sh"
        _run_local_job_file(run_dir, job_file)


def extract_umap_directory_v2(dumap_output_file):
    """Extract the UMAP results directory from 3D UMAP output logs."""
    dumap_output_file = Path(dumap_output_file)
    if not dumap_output_file.exists():
        raise FileNotFoundError(f"3D UMAP output file not found: {dumap_output_file}")

    content = dumap_output_file.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"Saving UMAP results to (.+)", content)
    if match:
        return match.group(1).strip()

    raise ValueError(f"Could not find UMAP directory in {dumap_output_file}")


def extract_catalog_settings_v2(dumap_config_file):
    """Extract the catalog path and whether duplicate object IDs should be collapsed."""
    dumap_config_file = Path(dumap_config_file)
    if not dumap_config_file.exists():
        raise FileNotFoundError(f"3D UMAP config file not found: {dumap_config_file}")

    config = _load_toml(dumap_config_file)

    filter_catalog = config.get("data_set", {}).get("filter_catalog")
    if filter_catalog:
        return str(filter_catalog), True

    legacy_path = config.get("data_set", {}).get("astropy_table")
    if legacy_path:
        return str(legacy_path), False

    for split_name in ("train", "infer", "validate"):
        split_config = config.get("data_request", {}).get(split_name)
        for dataset_wrapper in _iter_dataset_wrappers(split_config):
            dataset_config = dataset_wrapper["data"].get("dataset_config", {})
            if isinstance(dataset_config, dict):
                filter_catalog = dataset_config.get("filter_catalog")
                if filter_catalog:
                    return str(filter_catalog), True
                astropy_table = dataset_config.get("astropy_table")
                if astropy_table:
                    return str(astropy_table), False

    raise ValueError(f"Could not find filter_catalog or astropy_table in {dumap_config_file}")


def create_3d_viz_json_v2(x, y, base_directory=DEFAULT_BASE_DIRECTORY, id_column="object_id"):
    """Create a 3D visualization JSON file from 3D UMAP results."""
    if save_umap_json is None:
        raise ImportError("Could not import save_umap_json from hyrax. Activate a Hyrax environment before running.")

    run_dir = Path(base_directory) / f"run{x}"
    dumap_name = f"3dumap{x}_{y}"
    dumap_output = run_dir / f"{dumap_name}.txt"
    dumap_config = run_dir / f"{dumap_name}.toml"

    if not dumap_output.exists():
        raise FileNotFoundError(f"3D UMAP output file not found: {dumap_output}")
    if not dumap_config.exists():
        raise FileNotFoundError(f"3D UMAP config file not found: {dumap_config}")

    umap_results_dir = extract_umap_directory_v2(dumap_output)
    fits_table_path, keep_first_match_only = extract_catalog_settings_v2(dumap_config)

    viz_dir = Path(base_directory) / "3d_viz_files"
    viz_dir.mkdir(exist_ok=True)
    output_json = viz_dir / f"umap{x}_{y}.json"

    save_umap_json(
        results_dir=umap_results_dir,
        output_json=str(output_json),
        fits_table_path=fits_table_path,
        id_column=id_column,
        keep_first_match_only=keep_first_match_only,
    )
    print(f"Successfully created {output_json}")
    return str(output_json)


def create_3d_viz_json_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY, id_column="object_id"):
    """Create 3D visualization JSON files for multiple jobs."""
    results = []
    failed = []

    for y in y_values:
        try:
            output_json = create_3d_viz_json_v2(x, y, base_directory=base_directory, id_column=id_column)
            results.append(output_json)
        except Exception as exc:
            print(f"Failed to process 3dumap{x}_{y}: {exc}")
            failed.append(y)

    print("\n3D Visualization JSON Generation Summary:")
    print(f"  Successfully created: {len(results)} JSON files")
    print(f"  Failed: {len(failed)} files")
    if failed:
        print(f"  Failed job numbers: {failed}")

    return results


In [4]:
from pathlib import Path

BASE_DIR = "/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs"
RUN_NUMBER = 1
# JOB_NUMBER = 1

FILTER_CATALOG = "/Users/diegomiura/research/Hyrax-Research/test_dir_100images/split_images/catalog.fits"
# FILTER_CATALOG = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_le_120x120.fits" # Crop to [53,53]
# FILTER_CATALOG = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_gt_120x120.fits" # Crop to [121,121]
RUN_DIR = Path(BASE_DIR) / f"run{RUN_NUMBER}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
create_training_files_v2(
      run_number=RUN_NUMBER,
      filter_catalog=FILTER_CATALOG,
      crop_to=[53, 53]
    #   batch_size=128,
    #   epochs=20,
  )

Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_1.toml
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_1.sh
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_2.toml
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_2.sh
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_3.toml
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_3.sh
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_4.toml
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_4.sh
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_5.toml
Created /Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/train1_5.sh
Created /Users/diego

In [6]:
submit_training_jobs_v2(RUN_NUMBER, 1, 8, base_directory=BASE_DIR)

Running locally: bash train1_1.sh
Running locally: bash train1_2.sh
Running locally: bash train1_3.sh
Running locally: bash train1_4.sh
Running locally: bash train1_5.sh
Running locally: bash train1_6.sh
Running locally: bash train1_7.sh
Running locally: bash train1_8.sh


In [7]:
create_infer_scripts_batch_v2(RUN_NUMBER, [1,2,3,4,5,6,7,8], base_directory=BASE_DIR)

Created infer scripts for train1_1
Created infer scripts for train1_2
Created infer scripts for train1_3
Created infer scripts for train1_4
Created infer scripts for train1_5
Created infer scripts for train1_6
Created infer scripts for train1_7
Created infer scripts for train1_8


[(PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_1.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_1.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_2.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_2.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_3.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_3.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_4.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_4.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/infer1_5.toml'),
  PosixPath('/Users/diegomiura/research/H

In [8]:
submit_infer_jobs_v2(RUN_NUMBER, [1,2,3,4,5,6,7,8], base_directory=BASE_DIR)

Running locally: bash infer1_1.sh
Running locally: bash infer1_2.sh
Running locally: bash infer1_3.sh
Running locally: bash infer1_4.sh
Running locally: bash infer1_5.sh
Running locally: bash infer1_6.sh
Running locally: bash infer1_7.sh
Running locally: bash infer1_8.sh


In [9]:
create_udb_scripts_batch_v2(RUN_NUMBER, [1,2,3,4,5,6,7,8], base_directory=BASE_DIR)

Created UDB scripts for infer1_1
Created UDB scripts for infer1_2
Created UDB scripts for infer1_3
Created UDB scripts for infer1_4
Created UDB scripts for infer1_5
Created UDB scripts for infer1_6
Created UDB scripts for infer1_7
Created UDB scripts for infer1_8


[(PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_1.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_1.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_2.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_2.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_3.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_3.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_4.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_4.sh')),
 (PosixPath('/Users/diegomiura/research/Hyrax-Research/test_dir_100images/hyrax_runs/run1/udb1_5.toml'),
  PosixPath('/Users/diegomiura/research/Hyrax-Research/test

In [11]:
submit_udb_jobs_v2(RUN_NUMBER, [1,2,3,4,5,6,7,8], base_directory=BASE_DIR)

Running locally: bash udb1_1.sh
Running locally: bash udb1_2.sh
Running locally: bash udb1_3.sh
Running locally: bash udb1_4.sh
Running locally: bash udb1_5.sh
Running locally: bash udb1_6.sh
Running locally: bash udb1_7.sh
Running locally: bash udb1_8.sh
